In [ ]:
import json
from pathlib import Path

BASE = Path("/tmp/wav2vec2")
for jsonfile in BASE.glob("*.json"):
    id = jsonfile.stem
    with open(jsonfile, "r") as f, open(BASE / f"{id}.ctm", "w") as out:
        data = json.load(f)
        for item in data["chunks"]:
            text = item["text"].lower().strip()
            start = float(item["timestamp"][0])
            end = float(item["timestamp"][1])
            out.write(f"{id} 1 {start:.2f} {end - start:.2f} {text} 1.0\n")

In [2]:
BASE = Path("/tmp/whisper")
for jsonfile in BASE.glob("*.json"):
    id = jsonfile.stem.replace(".whisper", "")

    with open(jsonfile, "r") as f, open(BASE / f"{id}.txt", "w") as out:
        data = json.load(f)
        text = data["text"].strip()
        out.write(f"{id} {text}\n")

In [ ]:
!for ctm in /tmp/wav2vec2/*.ctm; do i=$(basename $ctm .ctm); python -m sync_asr.kaldi.align_ctm_ref --ref /tmp/whisper/$i.txt --hyp /tmp/wav2vec2/$i.ctm --output /tmp/$i.ctm;done

In [ ]:
BASE = Path("/tmp/pass1")
OUT = Path("/tmp/pass2")
OUT.mkdir(exist_ok=True)

OK = {
    "emtm": "MTM",
    "emteem": "MTM",
    "esokss": "SOS",
    "ämtm": "MTM",
    "ss": "S&S",
    "soss": "SOS",
}

def checker(a, b):
    a = a.lower().strip(",.?!;:")
    b = b.lower().strip(",.?!;:")
    if a == b:
        return True
    if a in OK and OK[a].lower() == b:
        return True
    return False

for ctmfile in BASE.glob("*.ctm"):
    id = ctmfile.stem
    with open(ctmfile, "r") as f, open(OUT / f"{id}.ctm", "w") as out:
        for line in f:
            line = line.strip()
            parts = line.split()
            wv = parts[4]
            wh = parts[6]
            code = parts[7]
            if code == "cor":
                out.write(line + "\n")
            elif code == "sub":
                if checker(wv, wh):
                    if parts[6] == "SOS":
                        parts[6] = "S&S"
                    parts[7] = "cor"
                    parts[4] = parts[6]
                    out.write(" ".join(parts) + "\n")
                else:
                    out.write(line + "\n")
            else:
                out.write(line + "\n")